# LoRA Fine-Tuning: Llama3.2-1B and Qwen2.5-14B on Hate Speech Data

This notebook fine-tunes two LLMs using LoRA (Low-Rank Adaptation) via Unsloth and TRL:
- `unsloth/Llama-3.2-1B-Instruct-bnb-4bit`
- `unsloth/Qwen2.5-14B-Instruct-bnb-4bit`

**Training configurations:**
- **Human only**: Fine-tune on 7 human-labeled datasets.
- **Synthetic (LGB/Vote/Mean)**: Fine-tune on OWS synthetic data labeled by the chosen ensemble method.
- **Human + Synthetic**: Combine human-labeled and LGB-labeled synthetic data.

**Input datasets**: `danghaidang-passau/Hate.2_labels_labeled` (synthetic) and
`danghaidang-passau/train_set` (human-labeled).

**Output**: LoRA adapter checkpoints saved locally; optionally uploaded to Hugging Face.
The `label_id` assignment at the label selection cell controls which ensemble strategy is used for training.

In [1]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

from sklearn.model_selection import train_test_split


/home/dangdang/miniconda3/envs/ros_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'trl'

## 2. Load Synthetic Dataset and Define Prompt Template

Load the OWS synthetic dataset (`danghaidang-passau/Hate.2_labels_labeled`) which contains the full
corpus annotated with per-model LLM probabilities and the ensemble label columns (`Lgb`, `Vote`, `Mean`).

The prompt template instructs the model to classify a comment as:
- `1` = Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech
- `2` = Neutral Speech

This template is used during both training (as the supervised fine-tuning format) and inference.

In [2]:

repo = "danghaidang-passau/HateOWS-dataset-LREC2026"
ds_annotated = load_dataset(repo, "annotatedOWS")    
df_2_label = ds_annotated["train"].to_pandas()

df = df_2_label
prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
2 for Neutral Speech.

Provide only the number corresponding to the category. Do not include any explanation or additional text.
Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Neutral speech?
\n"{comment}"\n
Your Answer:
'''



In [3]:
df.iloc[0]

text              Guten Tag, \r\n\r\nist das wahr, dass man Mine...
language                                                        deu
qwen_prob_1                                                     0.0
qwen_prob_2                                                     1.0
gemma_prob_1                                               0.000038
gemma_prob_2                                                    1.0
llama_prob_1                                                0.00038
llama_prob_2                                                    1.0
mistral_prob_1                                             0.000002
mistral_prob_2                                                  1.0
mean_prob_1                                                0.000105
mean_prob_2                                                     1.0
mean_label                                                        2
lgb_prob_1                                                 0.004639
lgb_prob_2                                      

## 3. Select Ensemble Label Strategy

To use a specific ensemble labeling strategy as the training signal for the synthetic data,
uncomment the corresponding line below:
- `df['Lgb']` — LightGBM meta-model labels (generally the best performing)
- `df['Mean']` — Mean average probability labels
- `df['Vote']` — Majority vote labels

In [ ]:
df['label_id'] = df['lgb_label']
# df['label_id'] = df['mean_label']
# df['label_id'] = df['vote_label']
df['label_id'].value_counts()

## 4. Load Human-Labeled Training Data

Load the human-annotated training set (`danghaidang-passau/train_set`) and filter it to the 7 datasets
used in the paper: HateXplain, GermEval19, ViHSD, GermEval21, US_election, Covid, and Sexism.
These datasets are used as the human-labeled baseline and as the basis for the Human-LGB combined training.

In [ ]:
repo = "danghaidang-passau/HateOWS-dataset-LREC2026"
ds_16 = load_dataset(repo, "16-val-train")       
human_df = ds_16["train"]


In [5]:
human_df = human_df[human_df.ds.isin(['HateXplain', 'GermEval19', 'ViHSD', 'GermEval21', 'US_election', 'Covid', 'Sexism'])].reset_index(drop=True)
human_df.ds.value_counts()

ds
HateXplain     15299
Sexism         10904
GermEval19      9698
ViHSD           8061
GermEval21      2071
US_election     1283
Covid           1282
Name: count, dtype: int64

In [6]:
human_df.shape[0]

48598

## 5. Define Helper Functions

**`process_task`**: Runs a batch through the model and extracts softmax probabilities for label tokens.
Only used for inference, not during LoRA training.

**`preprocess`**: Formats a (text, label) pair into the chat template for supervised fine-tuning.
The assistant's response is the label string ("1" or "2"), which becomes the SFT target token.

In [10]:
def process_task(texts, model, tokenizer, stop_token_id):
    encoding = tokenizer(texts, padding=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits  
    last_token_logits = logits[:, -1, :] 
    probabilities = torch.softmax(last_token_logits, dim=-1)
    indices = torch.tensor(stop_token_id)
    probs = []
    for i in indices:
        probs.append( probabilities[:, i].float().cpu().numpy())
    return probs

def preprocess(text, label, model_id, tokenizer):
    user_message_content = prompt_template.format(comment=text)
    user_message = {
        "role": "user",
        "content": user_message_content
    }

    assistant_message = {
        "role": "assistant",
        "content": str(label)
    }

    if "Qwen" in model_id:
        system_message =  {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant"}
    else:
        system_message =  {"role": "system", "content": "You are a helpful assistant"}
    if "gemma" in model_id:
        messages = [user_message, assistant_message]
        messages = tokenizer.apply_chat_template(messages, tokenize=False)

    else:
        messages = [system_message, user_message, assistant_message]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    messages = messages
    
    return messages


## 6. Configure Training Dataset Configuration

Set the label strategy (`label_id`) and choose the training data source:
- **Synthetic only**: `df = df_2_label` (OWS synthetic data with chosen ensemble labels)
- **Human only**: `df = human_df`
- **Human + Synthetic**: Use `pd.concat([df_2_label, human_df])` for combined training

The label selection cell sets which ensemble column (`Lgb`, `Mean`, or `Vote`) to use as ground truth.

In [ ]:
# Selecting label
df['label_id'] = df['lgb_label']
# df['label_id'] = df['mean_label']
# df['label_id'] = df['vote_label']

In [ ]:
# Training human dataset
df = human_df

# Companing human + synthetic data:
df = pd.concat([df, human_df]).reset_index(drop=True)

## 7. Load Model and Build Training Prompts

Load the base model (Llama3.2-1B or Qwen2.5-14B) in 4-bit quantization using Unsloth's
`FastLanguageModel`. Apply the chat template to all training samples using `preprocess` to create
the SFT-formatted prompt strings. Each formatted prompt ends with the assistant's label token
("1" or "2") which is the training target.

In [ ]:
model_id ="unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
model_id = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 500,
    dtype = getattr(torch, "bfloat16"),
)

df["prompt"] = human_df.apply(lambda row: preprocess(row["text"], row["label_id"],  model_id, tokenizer), axis=1)


In [12]:
print(df.prompt[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 30 Apr 2025

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

You are tasked with annotating speech. Your response must be a single valid number:
    1 for Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech,
    2 for Normal Speech.

    Provide only the number corresponding to the category. Do not include any explanation or additional text.
    Do you think the following comment is Hate/Offensive/Sexism/Toxic/Political/COVID-related Hate Speech, speech or Normal speech?
    
"Guten Tag, 

ist das wahr, dass man Minecraft irgendwie kostenlos haben kann?
Wenn ja, wie und wo?

LG"

    Your Answer:<|eot_id|><|start_header_id|>assistant<|end_header_id|>

2<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## 8. LoRA Fine-Tuning with SFT Trainer

Configure and run supervised fine-tuning using `SFTTrainer` from TRL. The LoRA adapters are applied
to the model's attention layers via Unsloth's `FastLanguageModel.get_peft_model`.

Key training settings:
- 1 epoch, cosine LR schedule, `bfloat16` precision
- Gradient accumulation over 8 steps with per-device batch size 8 (effective batch size 64)
- Max gradient norm 0.2, warmup ratio 0.1
- Checkpoints saved every 1,000 steps to `output_dir`



In [ ]:
# dataset = Dataset.from_pandas(df.sample(n=2000))
dataset = Dataset.from_pandas(df.sample(frac=1, random_state=def_seed))


dataset = dataset.shuffle(seed=42)


output_dir = "Lgb.Llama1B"
sft_config = SFTConfig(
    output_dir=output_dir, 
    eval_strategy="no", 
    do_eval=False,  
    optim="paged_adamw_8bit", 
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=8,  
    per_device_eval_batch_size=1,  
    log_level="debug",  
    save_steps=1000,
    logging_steps=200, 
    learning_rate=1e-5,  

    eval_steps=5000, 
    max_steps=-1, 
    num_train_epochs=1, 
    warmup_ratio=0.1, 
    lr_scheduler_type="cosine", 
    fp16=False,  
    bf16=True, 
    max_grad_norm=0.2, 
    gradient_checkpointing=True, 
    gradient_checkpointing_kwargs={'use_reentrant':False}, 

    dataset_text_field="prompt",
    max_seq_length=500, 
    packing=False,
    report_to=[],
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none", 
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = True, 
    loftq_config = None, 
)

model.gradient_checkpointing_enable()
tokenizer.pad_token = tokenizer.eos_token 

trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,  
        tokenizer=tokenizer, 
        args=sft_config, 
)

trainer.model.print_trainable_parameters()

trainer.train()


PyTorch: setting up devices
Unsloth: Already have LoRA adapters! We shall skip this step.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Using auto half precision backend


trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540


Currently training with a batch size of: 8
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 8 | Gradient Accumulation steps = 8
\        /    Total batch size = 64 | Total steps = 31
 "-____-"     Number of trainable parameters = 5,636,096


Step,Training Loss


Saving model checkpoint to Lgb.Llama1B/checkpoint-31
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--llama-3.2-1b-instruct-unsloth-bnb-4bit/snapshots/8eb7999dc1d92775e7b5f75754a29c3fbdb32723/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_qu

TrainOutput(global_step=31, training_loss=2.566320603893649, metrics={'train_runtime': 48.9296, 'train_samples_per_second': 40.875, 'train_steps_per_second': 0.634, 'total_flos': 3065217004634112.0, 'train_loss': 2.566320603893649, 'epoch': 0.992})